In [ ]:
import os
import csv
import time
from openai import OpenAI
import base64
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from google.colab import userdata
or_token = userdata.get('OR_Key_2')
print("Token loaded:", or_token[:10] + "...")

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=or_token,
)

In [ ]:
!unzip dataset.zip

In [ ]:
# Path to dataset and output CSV
dataset_path = "/content/dataset"   # e.g., "dataset/"
output_csv = "results_baseline.csv"

In [ ]:
# 🔹 Updated prompt: model will identify the rice disease or pest name
# You need to check and correct the disease and pest names manually to match with the original name after downloading CSV.
# For example: Model may produce output such as 'Bacterial Blight' instead of 'Bacterial Leaf Blight' or 'Rice Blast' instead of 'Leaf Blast'
prompt = """You are an agricultural expert specializing in rice.
Task:
Identify the rice disease or pest shown in the provided image.
Rules:
1) Carefully examine the image.
2) Identify the single most accurate rice disease or pest based only on visible symptoms.
3) You must output exactly one disease or pest name.
4) If the image is unclear, choose the closest matching disease or pest based on visible symptoms.
5) Do not provide any explanation, description, or extra text.

Output format:
<disease or pest name only>"""

# 🔹 Create CSV file if it doesn’t exist
if not os.path.exists(output_csv):
    with open(output_csv, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["file_name", "actual_class", "predicted_class"])

# 🔹 Loop through folders (each folder = actual class)
for class_name in os.listdir(dataset_path):
    class_folder = os.path.join(dataset_path, class_name)
    if not os.path.isdir(class_folder):
        continue

    counter = 1
    for file_name in os.listdir(class_folder):
        if not (file_name.lower().endswith((".jpg", ".jpeg", ".png"))):
            continue

        image_path = os.path.join(class_folder, file_name)
        print(f"Processing: {file_name} ...")

        # Read image as base64
        with open(image_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")

        # Send to Gemini model
        #google/gemini-2.5-flash
        #openai/gpt-5.2
        #qwen/qwen3.7-plus
        #meta-llama/llama-4-maverick
        try:
            completion = client.chat.completions.create(
            model="google/gemini-2.5-flash",
            temperature=0.0,
            top_p=1.0,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",  # Still use 'image_url', but with base64 data
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{b64}"
                            }
                        }
                    ]
                }
            ]
        )

            predicted_class = completion.choices[0].message.content.strip()

        except Exception as e:
            print(f"❌ Error with {file_name}: {e}")
            predicted_class = "Error"

        # Append result to CSV
        with open(output_csv, mode='a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([file_name, class_name, predicted_class])

        print(f"✅ {counter}. Done: {file_name} | Actual: {class_name} | Predicted: {predicted_class}")
        counter += 1

        # ⏳ Wait 2 seconds before next request
        #time.sleep(2)